# Lab 24-09 · Linear Budgets, Laser Sharing and Reliability Bounds

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import cpo

## Part 1 · Linear budget

Both host channels give $Q$ = 10. *Predict* the optical $Q$ at which the margin above KP4 falls to 2 dB.

In [2]:
qo = np.linspace(5, 12, 200)
m = [cpo.margin_db(cpo.linear_q([10, q, 10])) for q in qo]
q2 = optimize.brentq(lambda q: cpo.margin_db(cpo.linear_q([10, q, 10])) - 2.0, 4, 12)
print(f"margin reaches 2 dB at optical Q = {q2:.2f}")
plt.figure()
plt.plot(qo, m)
plt.axhline(2.0, color="0.5", ls=":")
plt.xlabel("optical segment Q"); plt.ylabel("margin above KP4 (dB)"); plt.grid(True)
plt.show()

margin reaches 2 dB at optical Q = 5.61


## Part 2 · Laser sharing

*Predict* the lanes per UHP and SHP source.

In [3]:
for name, dbm in (("UHP", 23), ("SHP", 26)):
    avail, n, nbin = cpo.lanes_per_laser(dbm, 2.5, 1.0, 3.69)
    print(f"{name} {dbm} dBm: {avail:.2f} dB to split, factor {n:.1f}, {nbin} lanes with binary splitting")

UHP 23 dBm: 15.81 dB to split, factor 38.1, 32 lanes with binary splitting
SHP 26 dBm: 18.81 dB to split, factor 76.0, 64 lanes with binary splitting


## Part 3 · Zero-failure tests

Set the true rate to the rule-of-three bound for one million link-hours. *Predict* the fraction of tests that see no failures.

In [4]:
T = 1e6
lam = -math.log(0.05) / T
print(f"simulated zero-failure fraction: {cpo.zero_failure_fraction(lam, T):.4f} (exact {math.exp(-lam * T):.4f})")
for r in (0, 1, 2):
    print(f"{r} failure(s) in 1 M h: 95% bound {cpo.fit_bound(T, r):.0f} FIT")
hours = np.logspace(5, 8.5, 100)
plt.figure()
plt.loglog(hours, [cpo.fit_bound(h) for h in hours])
plt.xlabel("failure-free link-hours"); plt.ylabel("95% upper bound (FIT)"); plt.grid(True, which="both")
plt.show()

simulated zero-failure fraction: 0.0504 (exact 0.0500)
0 failure(s) in 1 M h: 95% bound 2996 FIT
1 failure(s) in 1 M h: 95% bound 4744 FIT
2 failure(s) in 1 M h: 95% bound 6296 FIT


## Part 4 · Blast radius

*Predict* the repair time at which a CPO port loses no more than a pluggable port.

In [5]:
base = cpo.port_hours_lost(200, 1, 1.0, 1)
mttr = np.linspace(0.1, 24, 200)
ratio = [cpo.port_hours_lost(400, 8, t, 8) / base for t in mttr]
t_eq = optimize.brentq(lambda t: cpo.port_hours_lost(400, 8, t, 8) - base, 0.01, 24)
print(f"pluggable: {base:.2e} port-h per port-year; break-even CPO repair time {t_eq:.2f} h; ratio at 24 h {ratio[-1]:.0f}")
plt.figure()
plt.plot(mttr, ratio)
plt.xlabel("CPO repair time (h)"); plt.ylabel("port-hours lost, CPO / pluggable"); plt.grid(True)
plt.show()

pluggable: 1.75e-03 port-h per port-year; break-even CPO repair time 0.50 h; ratio at 24 h 48
